Environment Anchoring, Drive Mount, & Global Mappings

In [ ]:
# ==============================================================================
# PHASE 1: STACK SETUP & ENVIRONMENT ANCHORING
# ==============================================================================
import os
from google.colab import drive

# Mount Google Drive workspace
drive.mount('/content/drive')
STORAGE_PATH = '/content/drive/MyDrive/Sanskrit_NMT_Project'
os.makedirs(STORAGE_PATH, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Global Token Vocabulary Mappings
PAD_SYM, SOS_SYM, EOS_SYM, UNK_SYM = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
PAD_CODE, SOS_CODE, EOS_CODE, UNK_CODE = 0, 1, 2, 3

Mounted at /content/drive


System Hyperparameters & Text Processing Utilities

In [ ]:
# ==============================================================================
# PHASE 2: HIGH-CAPACITY ENGINE HYPERPARAMETERS
# ==============================================================================
import re
import time
import math
import random
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as opt

class NmtHyperParams:
    GLOBAL_SEED = 42
    TOKEN_EMBED_SIZE = 512
    RNN_CELL_SIZE = 512
    STACK_DEPTH = 3           # Reduced from 4 to prevent aggressive overfitting
    DROPOUT_PROB = 0.4        # Increased from 0.3 to heavily regularize training layers
    MINI_BATCH_SIZE = 32
    BASE_LR = 0.0003
    L2_REG = 1e-4             # Increased to 1e-4 for stronger weight decay penalties
    FORCE_RATIO = 0.5
    GRAD_CLIP_VAL = 1.0
    MAX_EPOCHS = 30
    EARLY_STOP_WINDOW = 6
    SEARCH_BEAM_SIZE = 5
    SEQUENCE_LIMIT = 50
    MIN_TOKEN_OCCURRENCE = 2  # Set to 2 to eliminate single-occurrence noise
    COMPUTE_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def enforce_deterministic_runtime(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

enforce_deterministic_runtime(NmtHyperParams.GLOBAL_SEED)

def sanitize_sanskrit_string(raw_string):
    return re.sub(r'\s+', ' ', str(raw_string)).strip()

def sanitize_english_string(raw_string):
    processed = str(raw_string).lower()
    processed = re.sub(r'[^a-z0-9\s.,!?]', '', processed)
    return re.sub(r'\s+', ' ', processed).strip()

def segment_sanskrit_tokens(text_blob): return text_blob.split()
def segment_english_tokens(text_blob): return text_blob.split()

CSV Data Cleaners, Vocabulary, & Active DataLoaders
Python

In [ ]:
# ==============================================================================
# PHASE 3: FILE ALIGNMENT & SYMLINK INTEGRATION
# ==============================================================================
from torch.utils.data import Dataset, DataLoader

DATA_MANIFEST = [
    'train_sa_10000.csv', 'train_en_10000.csv',
    'dev_sa_1000.csv', 'dev_en_1000.csv',
    'test_sa_1000.csv', 'test_en_1000.csv'
]

def initialize_empty_fallback_matrices():
    target_sets = [
        ('train_sa_10000.csv', 'train_en_10000.csv', 100),
        ('dev_sa_1000.csv', 'dev_en_1000.csv', 10),
        ('test_sa_1000.csv', 'test_en_1000.csv', 10)
    ]
    for sa_file, en_file, row_count in target_sets:
        if not os.path.exists(sa_file):
            pd.DataFrame({'Source_id': [str(idx) for idx in range(row_count)], 'Sentence_sa': ["भवतः नाम किम् ?" for _ in range(row_count)]}).to_csv(sa_file, index=False)
        if not os.path.exists(en_file):
            pd.DataFrame({'Source_id': [str(idx) for idx in range(row_count)], 'Sentence_en': ["what is your name ?" for _ in range(row_count)]}).to_csv(en_file, index=False)

for specific_file in DATA_MANIFEST:
    cloud_source = os.path.join(STORAGE_PATH, specific_file)
    if os.path.exists(cloud_source) and not os.path.exists(specific_file):
        os.symlink(cloud_source, specific_file)

initialize_empty_fallback_matrices()

DATA REFORMATTING

In [ ]:
# ==============================================================================
# PHASE 4 & 5: DATA REFORMATTING & VOCABULARY CORPUS BUILDER
# ==============================================================================
def compile_paired_dataframe(src_csv, trg_csv):
    left_df = pd.read_csv(src_csv)
    right_df = pd.read_csv(trg_csv)
    merged_df = pd.merge(left_df, right_df, on='Source_id', how='inner')
    merged_df.dropna(subset=['Sentence_sa', 'Sentence_en'], inplace=True)
    merged_df.drop_duplicates(subset=['Sentence_sa', 'Sentence_en'], inplace=True)
    return merged_df

training_set = compile_paired_dataframe('train_sa_10000.csv', 'train_en_10000.csv')
validation_set = compile_paired_dataframe('dev_sa_1000.csv', 'dev_en_1000.csv')
evaluation_set = compile_paired_dataframe('test_sa_1000.csv', 'test_en_1000.csv')

for target_dataframe in [training_set, validation_set, evaluation_set]:
    target_dataframe['Sentence_sa'] = target_dataframe['Sentence_sa'].apply(sanitize_sanskrit_string)
    target_dataframe['Sentence_en'] = target_dataframe['Sentence_en'].apply(sanitize_english_string)

class TextCorpusVocabulary:
    def __init__(self):
        self.token_to_index = {PAD_SYM: PAD_CODE, SOS_SYM: SOS_CODE, EOS_SYM: EOS_CODE, UNK_SYM: UNK_CODE}
        self.index_to_token = {PAD_CODE: PAD_SYM, SOS_CODE: SOS_SYM, EOS_CODE: EOS_SYM, UNK_CODE: UNK_SYM}
        self.token_frequencies = collections.Counter()

    def construct_vocabulary(self, array_of_sentences, tokenization_method, floor_limit):
        for line in array_of_sentences:
            self.token_frequencies.update(tokenization_method(line))
        next_assigned_id = 4
        for word_token, counter in self.token_frequencies.items():
            if counter >= floor_limit:
                self.token_to_index[word_token] = next_assigned_id
                self.index_to_token[next_assigned_id] = word_token
                next_assigned_id += 1

    def __len__(self): return len(self.token_to_index)
    def transform_to_indices(self, text_body, tokenization_method):
        return [self.token_to_index.get(token, UNK_CODE) for token in tokenization_method(text_body)]

source_vocab = TextCorpusVocabulary()
source_vocab.construct_vocabulary(training_set['Sentence_sa'].tolist(), segment_sanskrit_tokens, NmtHyperParams.MIN_TOKEN_OCCURRENCE)

target_vocab = TextCorpusVocabulary()
target_vocab.construct_vocabulary(training_set['Sentence_en'].tolist(), segment_english_tokens, NmtHyperParams.MIN_TOKEN_OCCURRENCE)

PIPELINE WRAPPERS

In [ ]:
# ==============================================================================
# PHASE 6: CUSTOM PYTORCH PIPELINE WRAPPERS
# ==============================================================================

class NmtDatasetContainer(Dataset):
    def __init__(self, core_df, src_vocab_obj, trg_vocab_obj, src_tokenizer, trg_tokenizer, ceiling_length=NmtHyperParams.SEQUENCE_LIMIT):
        self.underlying_data = core_df.reset_index(drop=True)
        self.src_vocab = src_vocab_obj
        self.trg_vocab = trg_vocab_obj
        self.src_tok = src_tokenizer
        self.trg_tok = trg_tokenizer
        self.max_seq_len = ceiling_length

    def __len__(self): return len(self.underlying_data)
    def __getitem__(self, sequence_idx):
        dataset_row = self.underlying_data.iloc[sequence_idx]
        numerical_src = self.src_vocab.transform_to_indices(dataset_row['Sentence_sa'], self.src_tok)[:self.max_seq_len - 2] + [EOS_CODE]
        numerical_trg = [SOS_CODE] + self.trg_vocab.transform_to_indices(dataset_row['Sentence_en'], self.trg_tok)[:self.max_seq_len - 2] + [EOS_CODE]
        return {'src_indices': torch.tensor(numerical_src, dtype=torch.long), 'trg_indices': torch.tensor(numerical_trg, dtype=torch.long), 'row_uid': str(dataset_row['Source_id'])}

def dynamic_padding_collate_fn(data_batch):
    source_tensor_list = [element['src_indices'] for element in data_batch]
    target_tensor_list = [element['trg_indices'] for element in data_batch]
    unique_identifiers = [element['row_uid'] for element in data_batch]

    actual_src_lengths = torch.tensor([len(sequence) for sequence in source_tensor_list], dtype=torch.long)
    actual_trg_lengths = torch.tensor([len(sequence) for sequence in target_tensor_list], dtype=torch.long)

    padded_src_matrix = nn.utils.rnn.pad_sequence(source_tensor_list, batch_first=True, padding_value=PAD_CODE)
    padded_trg_matrix = nn.utils.rnn.pad_sequence(target_tensor_list, batch_first=True, padding_value=PAD_CODE)

    actual_src_lengths, sorting_permutation = actual_src_lengths.sort(descending=True)
    padded_src_matrix = padded_src_matrix[sorting_permutation]
    padded_trg_matrix = padded_trg_matrix[sorting_permutation]
    actual_trg_lengths = actual_trg_lengths[sorting_permutation]
    unique_identifiers = [unique_identifiers[pointer] for pointer in sorting_permutation]

    return {'padded_src': padded_src_matrix, 'src_lens': actual_src_lengths, 'padded_trg': padded_trg_matrix, 'trg_lens': actual_trg_lengths, 'item_uids': unique_identifiers}

train_data_loader = DataLoader(NmtDatasetContainer(training_set, source_vocab, target_vocab, segment_sanskrit_tokens, segment_english_tokens), batch_size=NmtHyperParams.MINI_BATCH_SIZE, shuffle=True, collate_fn=dynamic_padding_collate_fn)
valid_data_loader = DataLoader(NmtDatasetContainer(validation_set, source_vocab, target_vocab, segment_sanskrit_tokens, segment_english_tokens), batch_size=NmtHyperParams.MINI_BATCH_SIZE, shuffle=False, collate_fn=dynamic_padding_collate_fn)
test_data_loader = DataLoader(NmtDatasetContainer(evaluation_set, source_vocab, target_vocab, segment_sanskrit_tokens, segment_english_tokens), batch_size=NmtHyperParams.MINI_BATCH_SIZE, shuffle=False, collate_fn=dynamic_padding_collate_fn)

Neural Architecture Layers & Network Classes
Python

In [ ]:
# ==============================================================================
# PHASE 7: PARA-DEEP HIGH-CAPACITY NEURAL NETWORK BLOCKS
# ==============================================================================
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class ParaDeepFeatureEncoder(nn.Module):
    def __init__(self, vocab_bound, embedding_dimension, recurrent_dimension, layer_count, dropout_factor):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_bound, embedding_dimension, padding_idx=PAD_CODE)
        self.recurrent_layers = nn.ModuleList([
            nn.LSTM(embedding_dimension if step == 0 else recurrent_dimension * 2, recurrent_dimension, num_layers=1, bidirectional=True, batch_first=True)
            for step in range(layer_count)
        ])
        self.dropout_layer = nn.Dropout(dropout_factor)
        self.linear_transform_h = nn.Linear(recurrent_dimension * 2, recurrent_dimension)
        self.linear_transform_c = nn.Linear(recurrent_dimension * 2, recurrent_dimension)

    def forward(self, input_batch, sequence_lengths):
        transformed_embeddings = self.dropout_layer(self.token_embedding(input_batch))
        hidden_states_accumulator, cell_states_accumulator = [], []

        for lstm_block in self.recurrent_layers:
            packed_tensor_sequence = pack_padded_sequence(transformed_embeddings, sequence_lengths.cpu(), batch_first=True, enforce_sorted=True)
            packed_recurrent_output, (h_state, c_state) = lstm_block(packed_tensor_sequence)
            unpacked_recurrent_output, _ = pad_packed_sequence(packed_recurrent_output, batch_first=True)

            concatenated_h = torch.cat((h_state[0], h_state[1]), dim=1)
            concatenated_c = torch.cat((c_state[0], c_state[1]), dim=1)
            hidden_states_accumulator.append(self.linear_transform_h(concatenated_h))
            cell_states_accumulator.append(self.linear_transform_c(concatenated_c))
            transformed_embeddings = self.dropout_layer(unpacked_recurrent_output)

        return unpacked_recurrent_output, (torch.stack(hidden_states_accumulator, dim=0), torch.stack(cell_states_accumulator, dim=0))

class AdvancedBahdanauAlignment(nn.Module):
    def __init__(self, internal_hidden_dim):
        super().__init__()
        self.encoder_projection = nn.Linear(internal_hidden_dim * 2, internal_hidden_dim, bias=False)
        self.decoder_projection = nn.Linear(internal_hidden_dim, internal_hidden_dim, bias=False)
        self.score_v_vector = nn.Linear(internal_hidden_dim, 1, bias=False)

    def forward(self, dec_hidden_state, enc_output_matrix, padding_mask):
        projected_decoder = self.decoder_projection(dec_hidden_state).unsqueeze(1)
        projected_encoder = self.encoder_projection(enc_output_matrix)
        alignment_energy = self.score_v_vector(torch.tanh(projected_encoder + projected_decoder)).squeeze(2)
        if padding_mask is not None:
            alignment_energy = alignment_energy.masked_fill(padding_mask, -1e10)
        attention_weights_distribution = torch.softmax(alignment_energy, dim=1)
        computed_context_vector = torch.bmm(attention_weights_distribution.unsqueeze(1), enc_output_matrix).squeeze(1)
        return computed_context_vector, attention_weights_distribution

class ParaDeepFeatureDecoder(nn.Module):
    def __init__(self, vocab_bound, embedding_dimension, recurrent_dimension, layer_count, dropout_factor):
        super().__init__()
        self.vocabulary_size_limit = vocab_bound
        self.token_embedding = nn.Embedding(vocab_bound, embedding_dimension, padding_idx=PAD_CODE)
        self.alignment_attention = AdvancedBahdanauAlignment(recurrent_dimension)
        self.recurrent_engine = nn.LSTM(embedding_dimension + recurrent_dimension * 2, recurrent_dimension, num_layers=layer_count, dropout=dropout_factor if layer_count > 1 else 0.0, batch_first=True)
        self.classification_out = nn.Linear(recurrent_dimension + recurrent_dimension * 2 + embedding_dimension, vocab_bound)
        self.dropout_layer = nn.Dropout(dropout_factor)

    def forward(self, individual_token_input, tracking_hidden, encoder_matrices, padding_mask):
        embedded_token = self.dropout_layer(self.token_embedding(individual_token_input.unsqueeze(1)))
        context_vector, alignment_weights = self.alignment_attention(tracking_hidden[0][-1], encoder_matrices, padding_mask)
        combined_recurrent_input = torch.cat((embedded_token, context_vector.unsqueeze(1)), dim=2)
        recurrent_output, tracking_hidden = self.recurrent_engine(combined_recurrent_input, tracking_hidden)
        token_prediction_logits = self.classification_out(torch.cat((recurrent_output.squeeze(1), context_vector, embedded_token.squeeze(1)), dim=1))
        return token_prediction_logits, tracking_hidden, alignment_weights

class UnifiedParaDeepSeq2Seq(nn.Module):
    def __init__(self, structural_encoder, structural_decoder):
        super().__init__()
        self.encoder_module = structural_encoder
        self.decoder_module = structural_decoder

    def compile_padding_mask(self, base_matrix): return (base_matrix == PAD_CODE)

    def forward(self, source_padded_batch, source_lengths, target_padded_batch, active_forcing_ratio=0.5):
        batch_capacity, max_target_span = source_padded_batch.shape[0], target_padded_batch.shape[1]
        raw_predictions_tensor = torch.zeros(batch_capacity, max_target_span, self.decoder_module.vocabulary_size_limit).to(NmtHyperParams.COMPUTE_DEVICE)
        encoder_tensor_outputs, recurrent_hidden = self.encoder_module(source_padded_batch, source_lengths)
        padding_mask = self.compile_padding_mask(source_padded_batch)
        current_decoder_input = target_padded_batch[:, 0]

        for time_step in range(1, max_target_span):
            prediction_logits, recurrent_hidden, _ = self.decoder_module(current_decoder_input, recurrent_hidden, encoder_tensor_outputs, padding_mask)
            raw_predictions_tensor[:, time_step, :] = prediction_logits
            is_forcing_active = random.random() < active_forcing_ratio
            current_decoder_input = target_padded_batch[:, time_step] if is_forcing_active else prediction_logits.argmax(1)
        return raw_predictions_tensor

Network Training, Beam Decoding, & Modern Evaluation Metrics

In [ ]:
# ==============================================================================
# PHASE 8 & 9: HIGH-CAPACITY INITIALIZATION & LOSS OPTIMIZATION LOOPS
# ==============================================================================
print("====== INITIALIZING HIGH-CAPACITY MODEL INFRASTRUCTURE ======")
network_encoder = ParaDeepFeatureEncoder(len(source_vocab), NmtHyperParams.TOKEN_EMBED_SIZE, NmtHyperParams.RNN_CELL_SIZE, NmtHyperParams.STACK_DEPTH, NmtHyperParams.DROPOUT_PROB)
network_decoder = ParaDeepFeatureDecoder(len(target_vocab), NmtHyperParams.TOKEN_EMBED_SIZE, NmtHyperParams.RNN_CELL_SIZE, NmtHyperParams.STACK_DEPTH, NmtHyperParams.DROPOUT_PROB)
global_nmt_network = UnifiedParaDeepSeq2Seq(network_encoder, network_decoder).to(NmtHyperParams.COMPUTE_DEVICE)

calculated_total_parameters = sum(param.numel() for param in global_nmt_network.parameters())
print(f"Total Structural Model Parameters: {calculated_total_parameters:,}")

optimization_loss_function = nn.CrossEntropyLoss(ignore_index=PAD_CODE)
network_optimizer = opt.Adam(global_nmt_network.parameters(), lr=NmtHyperParams.BASE_LR, weight_decay=NmtHyperParams.L2_REG)
learning_rate_scheduler = opt.lr_scheduler.ReduceLROnPlateau(network_optimizer, mode='min', factor=0.5, patience=2)

absolute_minimal_loss = float('inf')
no_improvement_epochs_counter = 0
historical_train_losses, historical_valid_losses = [], []

print("\nTriggering Optimization Routine...")
for active_epoch in range(NmtHyperParams.MAX_EPOCHS):
    global_nmt_network.train()
    cumulative_train_loss = 0
    for batch_data in train_data_loader:
        src_matrix = batch_data['padded_src'].to(NmtHyperParams.COMPUTE_DEVICE)
        src_len_vector = batch_data['src_lens'].to(NmtHyperParams.COMPUTE_DEVICE)
        trg_matrix = batch_data['padded_trg'].to(NmtHyperParams.COMPUTE_DEVICE)

        network_optimizer.zero_grad()
        output_logits = global_nmt_network(src_matrix, src_len_vector, trg_matrix, NmtHyperParams.FORCE_RATIO)

        step_loss = optimization_loss_function(output_logits[:, 1:].reshape(-1, output_logits.shape[-1]), trg_matrix[:, 1:].reshape(-1))
        step_loss.backward()
        nn.utils.clip_grad_norm_(global_nmt_network.parameters(), NmtHyperParams.GRAD_CLIP_VAL)
        network_optimizer.step()
        cumulative_train_loss += step_loss.item()

    global_nmt_network.eval()
    cumulative_valid_loss = 0
    with torch.no_grad():
        for batch_data in valid_data_loader:
            src_matrix = batch_data['padded_src'].to(NmtHyperParams.COMPUTE_DEVICE)
            src_len_vector = batch_data['src_lens'].to(NmtHyperParams.COMPUTE_DEVICE)
            trg_matrix = batch_data['padded_trg'].to(NmtHyperParams.COMPUTE_DEVICE)

            output_logits = global_nmt_network(src_matrix, src_len_vector, trg_matrix, 0.0)
            cumulative_valid_loss += optimization_loss_function(output_logits[:, 1:].reshape(-1, output_logits.shape[-1]), trg_matrix[:, 1:].reshape(-1)).item()

    mean_epoch_train_loss = cumulative_train_loss / len(train_data_loader)
    mean_epoch_valid_loss = cumulative_valid_loss / len(valid_data_loader)
    historical_train_losses.append(mean_epoch_train_loss)
    historical_valid_losses.append(mean_epoch_valid_loss)
    learning_rate_scheduler.step(mean_epoch_valid_loss)

    print(f"Epoch Slot: {active_epoch+1:02} | High-Cap Train Loss: {mean_epoch_train_loss:.4f} | High-Cap Valid Loss: {mean_epoch_valid_loss:.4f}")

    if mean_epoch_valid_loss < absolute_minimal_loss:
        absolute_minimal_loss = mean_epoch_valid_loss
        torch.save(global_nmt_network.state_dict(), os.path.join(STORAGE_PATH, 'best_high_cap_para_deep.pt'))
        no_improvement_epochs_counter = 0
    else:
        no_improvement_epochs_counter += 1
        if no_improvement_epochs_counter >= NmtHyperParams.EARLY_STOP_WINDOW:
            print("System early stopping condition met. Terminating updates.")
            break




====== INITIALIZING HIGH-CAPACITY MODEL INFRASTRUCTURE ======
Total Structural Model Parameters: 48,839,841

Triggering Optimization Routine...
Epoch Slot: 01 | High-Cap Train Loss: 6.1479 | High-Cap Valid Loss: 5.6641
Epoch Slot: 02 | High-Cap Train Loss: 5.6324 | High-Cap Valid Loss: 5.5938
Epoch Slot: 03 | High-Cap Train Loss: 5.4311 | High-Cap Valid Loss: 5.5072
Epoch Slot: 04 | High-Cap Train Loss: 5.2808 | High-Cap Valid Loss: 5.5097
Epoch Slot: 05 | High-Cap Train Loss: 5.1704 | High-Cap Valid Loss: 5.4673
Epoch Slot: 06 | High-Cap Train Loss: 5.0561 | High-Cap Valid Loss: 5.4291
Epoch Slot: 07 | High-Cap Train Loss: 4.9585 | High-Cap Valid Loss: 5.4202
Epoch Slot: 08 | High-Cap Train Loss: 4.8749 | High-Cap Valid Loss: 5.4267
Epoch Slot: 09 | High-Cap Train Loss: 4.7743 | High-Cap Valid Loss: 5.3906
Epoch Slot: 10 | High-Cap Train Loss: 4.7000 | High-Cap Valid Loss: 5.3704
Epoch Slot: 11 | High-Cap Train Loss: 4.6290 | High-Cap Valid Loss: 5.3670
Epoch Slot: 12 | High-Cap Train

BEAM DECODER WITH DYNAMIC MATRIX CLIP

In [ ]:
# ==============================================================================
# PHASE 10: ANTI-CRASH BEAM DECODER WITH DYNAMIC MATRIX CLIP
# ==============================================================================
def execute_safe_beam_decoding(nmt_model, padded_src_row, tracking_len, beam_width=NmtHyperParams.SEARCH_BEAM_SIZE, target_limit=NmtHyperParams.SEQUENCE_LIMIT):
    nmt_model.eval()
    with torch.no_grad():
        enc_output_tensors, recurrent_hidden = nmt_model.encoder_module(padded_src_row, tracking_len)
        active_sequence_span = enc_output_tensors.shape[1]
        padding_mask = nmt_model.compile_padding_mask(padded_src_row)[:, :active_sequence_span]

        vocab_cardinality = nmt_model.decoder_module.vocabulary_size_limit
        operational_beam_span = min(beam_width, vocab_cardinality)
        tracked_beam_candidates = [(0.0, [SOS_CODE], recurrent_hidden)]

        for _ in range(target_limit):
            successor_options = []
            completed_sequences_count = 0

            for accumulated_score, sequence_history, alternative_hidden in tracked_beam_candidates:
                if sequence_history[-1] == EOS_CODE:
                    successor_options.append((accumulated_score, sequence_history, alternative_hidden))
                    completed_sequences_count += 1
                    continue

                decoder_input_tensor = torch.tensor([sequence_history[-1]], dtype=torch.long).to(NmtHyperParams.COMPUTE_DEVICE)
                prediction_logits, updated_hidden, _ = nmt_model.decoder_module(decoder_input_tensor, alternative_hidden, enc_output_tensors, padding_mask)
                probability_logs = torch.log_softmax(prediction_logits, dim=1).squeeze(0)
                highest_log_scores, matching_indices = probability_logs.topk(operational_beam_span)

                for step in range(operational_beam_span):
                    successor_options.append((
                        accumulated_score + highest_log_scores[step].item(),
                        sequence_history + [matching_indices[step].item()],
                        updated_hidden
                    ))

            tracked_beam_candidates = sorted(successor_options, key=lambda array_pair: array_pair[0], reverse=True)[:operational_beam_span]
            if completed_sequences_count == operational_beam_span:
                break

        filtered_indices = [idx for idx in tracked_beam_candidates[0][1] if idx not in (SOS_CODE, EOS_CODE, PAD_CODE)]
        return [target_vocab.index_to_token.get(idx, UNK_SYM) for idx in filtered_indices]

# ====================================================================
# PHASE 11: GENERATION LOOP & EVALUATION PIPELINE
# ====================================================================

In [ ]:
print("\nInitiating Test Set Translation Computations...")
global_nmt_network.load_state_dict(torch.load(os.path.join(STORAGE_PATH, 'best_high_cap_para_deep.pt')))
inference_timer_start = time.time()

computed_hypotheses, true_references, saved_submission_rows = [], [], []

for batch_data in test_data_loader:
    src_matrix = batch_data['padded_src'].to(NmtHyperParams.COMPUTE_DEVICE)
    src_len_vector = batch_data['src_lens'].to(NmtHyperParams.COMPUTE_DEVICE)
    trg_matrix = batch_data['padded_trg']
    item_uids = batch_data['item_uids']

    for row_idx in range(src_matrix.shape[0]):
        inferred_tokens = execute_safe_beam_decoding(global_nmt_network, src_matrix[row_idx].unsqueeze(0), src_len_vector[row_idx].unsqueeze(0))
        computed_hypotheses.append(inferred_tokens)
        gold_standard_tokens = [target_vocab.index_to_token.get(idx.item(), UNK_SYM) for idx in trg_matrix[row_idx] if idx.item() not in (SOS_CODE, EOS_CODE, PAD_CODE)]
        true_references.append([gold_standard_tokens])
        saved_submission_rows.append({'Source_id': item_uids[row_idx], 'Sentence_en': " ".join(inferred_tokens)})

total_elapsed_inference_seconds = time.time() - inference_timer_start

# Install and instantiate libraries
!pip install -q datasets evaluate sentencepiece accelerate bert-score scikit-learn
import evaluate
from bert_score import score as calculate_bertscore

# FIXED: Convert lists of individual word tokens back into full string sentences
hypothesis_sentence_strings = [" ".join(tokens_list) if tokens_list else "empty" for tokens_list in computed_hypotheses]
reference_sentence_strings = [" ".join(wrapped_list[0]) if wrapped_list[0] else "empty" for wrapped_list in true_references]

# Compute Hugging Face BLEU score smoothly using sentence strings
bleu_module = evaluate.load("bleu")
hf_bleu_results = bleu_module.compute(predictions=hypothesis_sentence_strings, references=reference_sentence_strings)
calculated_bleu = hf_bleu_results["bleu"]

# Compute BERTScore configurations
try:
    _, _, f1_raw = calculate_bertscore(hypothesis_sentence_strings, reference_sentence_strings, model_type="bert-base-uncased", lang="en", rescale_with_baseline=False)
    raw_bertscore = f1_raw.mean().item()

    _, _, f1_rescaled = calculate_bertscore(hypothesis_sentence_strings, reference_sentence_strings, model_type="bert-base-uncased", lang="en", rescale_with_baseline=True)
    rescaled_bertscore = f1_rescaled.mean().item()
except Exception as error_context:
    raw_bertscore, rescaled_bertscore = 0.0, 0.0

print("\n" + "="*60)
print(f"BLEU Score              : {calculated_bleu:.4f}")
print(f"BERTScore Unscaled (F1) : {raw_bertscore:.4f}")
print(f"BERTScore Scaled (F1)   : {rescaled_bertscore:.4f}")
print(f"Inference Time (Test)   : {total_elapsed_inference_seconds:.2f} sec")
print(f"Total Parameters        : {calculated_total_parameters:,}")
print(f"Trainable Parameters    : {calculated_total_parameters:,}")
print("="*60)

# Export plots
plt.figure(figsize=(6, 4))
plt.plot(historical_train_losses, label='Loss (Train)')
plt.plot(historical_valid_losses, label='Loss (Validation)')
plt.title('High-Capacity Processing Profile')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig(os.path.join(STORAGE_PATH, 'loss_history.png'))
plt.close()

# Export submission csv
compiled_records_df = pd.DataFrame(saved_submission_rows)
reference_test_layout = pd.read_csv('test_sa_1000.csv')
final_submission_dataframe = reference_test_layout[['Source_id']].astype(str).merge(compiled_records_df, on='Source_id', how='left').fillna("")
final_submission_dataframe.to_csv('submission.csv', index=False)
final_submission_dataframe.to_csv(os.path.join(STORAGE_PATH, 'submission.csv'), index=False)


Initiating Test Set Translation Computations...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.6 MB/s eta 0:00:00


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BLEU Score              : 0.1388
BERTScore Unscaled (F1) : 0.5748
BERTScore Scaled (F1)   : 0.3436
Inference Time (Test)   : 72.98 sec
Total Parameters        : 48,839,841
Trainable Parameters    : 48,839,841
